# Train gpt-from-scratch on Google Colab

This notebook trains the GPT built in [gpt-from-scratch](https://github.com/VictorMerk/gpt-from-scratch) end to end on a free Google Colab GPU, then samples text and evaluates the result. Everything is implemented from scratch in PyTorch: causal attention, KV-cache decoding, weight tying, BPE, and the LR schedule.

**Before running anything:** go to *Runtime > Change runtime type* and select **T4 GPU** as the hardware accelerator.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/VictorMerk/gpt-from-scratch.git
%cd gpt-from-scratch
!pip install -e . -q

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

from gpt_from_scratch.model import GPT_PRESETS

print(GPT_PRESETS)

## Full run (~30-60 min on T4)

Trains the default model (6 layers, 6 heads, 384 dims, block size 256) for 5000 iterations with a short LR warmup and bf16 mixed precision. This is the configuration comparable in size to nanoGPT's shakespeare_char model.

In [ ]:
!gpt-from-scratch-train --device cuda --max-iters 5000 --lr-warmup-iters 100 --dtype bfloat16

## Fast preview run (~5 min)

If you just want to see it learn, train a smaller model first: 4 layers, 4 heads, 256 dims, shorter context, fewer iterations. The resulting checkpoint is fully compatible with the sampling and evaluation commands below.

In [ ]:
!gpt-from-scratch-train --device cuda --n-layer 4 --n-head 4 --n-embd 256 --batch-size 32 --block-size 128 --max-iters 2000 --dtype bfloat16

In [ ]:
!gpt-from-scratch-sample --prompt "ROMEO:" --max-new-tokens 500 --temperature 0.8 --top-k 200

In [ ]:
!gpt-from-scratch-evaluate --checkpoint checkpoints/checkpoint.pt --max-batches 20

## Optional: save your checkpoint to Google Drive

Colab disks are wiped when the session ends. Mount Drive and copy the checkpoint out if you want to keep it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!cp checkpoints/checkpoint.pt /content/drive/MyDrive/